In [ ]:
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("src\\segment.py")

from segment import load_image, segment, draw_result

# --- paths -----------------------------------------------------------
# relative to the project root (CP467-fruit-recognition); walks up if the notebook is in a subfolder
root = Path.cwd()
while not (root / "data").exists() and root != root.parent:
    root = root.parent

TEST_DIR = root / "data" / "fruits-360_100x100" / "fruits-360" / "Test"
FIGURES_DIR = root / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

# --- collect all images (folder name = label, e.g. "Apple 5") --------
all_paths = sorted(TEST_DIR.rglob("*.jpg"))
print(f"{len(all_paths)} images, {len({p.parent.name for p in all_paths})} classes")

# --- pick 8 random images --------------------------------------------
rng = np.random.default_rng(42)          # change seed for a different sample
picks = rng.choice(len(all_paths), size=8, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(14, 7.5))

for ax, idx in zip(axes.ravel(), picks):
    path = all_paths[idx]
    label = path.parent.name

    img = load_image(path)
    mask, contour = segment(img)

    if contour is None:                  # segmentation found nothing
        vis = cv2.resize(img, None, fx=3, fy=3, interpolation=cv2.INTER_NEAREST)
        title = f"{label} (no contour)"
    else:
        vis = draw_result(img, contour, label="")   # outline + rotated box
        title = label

    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=11)
    ax.axis("off")

fig.suptitle("Segmentation check: 8 random dev images (green = contour, magenta = rotated bounding box)",
             fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "segmentation_check.png", dpi=200, bbox_inches="tight")
plt.show()
